# Teste do Tacotron 2 + WaveGlow Original da NVIDIA (LJSpeech)

Sanity check com os modelos **oficiais pré-treinados da NVIDIA** (Tacotron2 + WaveGlow), ambos treinados no **LJSpeech** — baixados diretamente via `torch.hub` (`pretrained=True`), sem depender de nenhum checkpoint local. Serve como referência de "estado da arte de fábrica" para comparar com os modelos Tacotron2-VAE treinados neste projeto (VERBO).

In [1]:
import os
import torch
import IPython.display as ipd
import numpy as np
import sys
from pathlib import Path

# Usamos GPU quando disponível. (Forçar CPU via CUDA_VISIBLE_DEVICES='' quebra o
# carregamento do checkpoint oficial da NVIDIA: os tensores foram salvos com
# localização CUDA, e o `torch.load` interno do entrypoint do hub não remapeia
# para CPU quando o dispositivo está completamente indisponível.)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando device: {device}")

Usando device: cuda


In [2]:
# Carregar o Tacotron 2 oficial da NVIDIA, com pesos pré-treinados baixados direto
# via torch.hub (pretrained=True) — não depende de nenhum checkpoint local.
tacotron2 = torch.hub.load(
    'NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tacotron2',
    model_math='fp32', pretrained=True, trust_repo=True,
)
tacotron2 = tacotron2.to(device).eval()
print("Tacotron 2 (NVIDIA, pré-treinado) carregado com sucesso!")

Using cache found in /home/al.richard.viana/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub
/home/al.richard.viana/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub/PyTorch/Classification/ConvNets/image_classification/models/common.py:13: UserWarning: pytorch_quantization module not found, quantization will not be available
  warnings.warn(
/home/al.richard.viana/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub/PyTorch/Classification/ConvNets/image_classification/models/efficientnet.py:17: UserWarning: pytorch_quantization module not found, quantization will not be available
  warnings.warn(


/home/al.richard.viana/Documentos/ml2_final_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/al.richard.viana/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub/PyTorch/SpeechSynthesis/Tacotron2/tacotron2/entrypoints.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimenta

Tacotron 2 (NVIDIA, pré-treinado) carregado com sucesso!


In [3]:
# Carregar o WaveGlow oficial da NVIDIA, também com pesos baixados via torch.hub.
waveglow = torch.hub.load(
    'NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow',
    model_math='fp32', pretrained=True, trust_repo=True,
)
waveglow = waveglow.remove_weightnorm(waveglow)
waveglow = waveglow.to(device).eval()
print('WaveGlow (NVIDIA, pré-treinado) carregado com sucesso!')

Using cache found in /home/al.richard.viana/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub
/home/al.richard.viana/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub/PyTorch/SpeechSynthesis/Tacotron2/waveglow/entrypoints.py:96: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full cont

/home/al.richard.viana/Documentos/ml2_final_project/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


WaveGlow (NVIDIA, pré-treinado) carregado com sucesso!


In [4]:
# 4. Text Processing
# Para usar os cleaners originais (english_cleaners), importamos direto do
# repositório que o torch.hub acabou de clonar na pasta de cache.
hub_dir = Path(torch.hub.get_dir()) / 'NVIDIA_DeepLearningExamples_torchhub' / 'PyTorch' / 'SpeechSynthesis' / 'Tacotron2'
if str(hub_dir) not in sys.path:
    sys.path.append(str(hub_dir))

from tacotron2.text import text_to_sequence

texto = "Hello! This is a test of the pre-trained NVIDIA Tacotron 2 model."
print(f"Texto: {texto}")

# Converter texto em sequência numérica de tokens.
# `torch.autograd.Variable` está deprecado desde o PyTorch 0.4 (tensores já
# suportam autograd nativamente) — usamos o tensor diretamente.
sequence = np.array(text_to_sequence(texto, ['english_cleaners']))[None, :]
sequence = torch.from_numpy(sequence).to(device).long()

Texto: Hello! This is a test of the pre-trained NVIDIA Tacotron 2 model.


In [5]:
# 5. Geracao e Audio
with torch.no_grad():
    # Passar texto pelo Tacotron2
    mel_outputs_postnet, mel_lengths, alignments = tacotron2.infer(sequence, torch.LongTensor([sequence.size(1)]).to(device))
    
    # Passar Espectrograma Mel pelo WaveGlow para gerar audio
    audio = waveglow.infer(mel_outputs_postnet, sigma=0.666)

audio_numpy = audio[0].data.cpu().numpy()

# 6. Tocar o Audio no Jupyter
ipd.Audio(audio_numpy, rate=22050)